In [ ]:
import torch
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt

In [ ]:
torch.cuda.is_available()      # True 나와야 함
torch.__version__              # 2.x.x+cu121 처럼 cu 붙어 있어야 함

In [ ]:
toTensor = torchvision.transforms.ToTensor()

cifar10 = torchvision.datasets.CIFAR10("./cifar10/", train=True, download=False)
cifar10_val = torchvision.datasets.CIFAR10("./cifar10_val/", train=False, download=False)

imgs = torch.stack([toTensor(img) for img, _ in cifar10])
mean = imgs.mean(dim=(0,2,3))
std = imgs.std(dim=(0,2,3))

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(
        mean=mean,
        std=std
    )
])

cifar10.transform = transform
cifar10_val.transform = transform

train_cifar10 = torch.utils.data.DataLoader(cifar10, batch_size=512, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)
val_cifar10 = torch.utils.data.DataLoader(cifar10_val, batch_size=512, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)


In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=5, padding=2), # 16x32x32
    nn.Tanh(),
    nn.MaxPool2d(2),
    nn.Conv2d(16,8, kernel_size=5, padding=2), # 8x16x16
    nn.Tanh(),
    nn.MaxPool2d(2), # 8x8x8
    nn.Flatten(1,-1),
    nn.Linear(512,32),
    nn.Tanh(),
    nn.Linear(32,10),
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

In [ ]:
epoch = 100
batch_size = 512

for n in range(epoch):
    loss = 0

    for imgs, labels in train_cifar10:
        imgs, labels = imgs.to(device), labels.to(device)
        p = model(imgs)
        _, out = torch.max(p, dim=1)
        loss = loss_fn(p, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"loss : {loss}")

In [ ]:
total = 0
count = 0

with torch.no_grad():
    for imgs, labels in val_cifar10:
        imgs, labels = imgs.to(device), labels.to(device)
        p = model(imgs)
        _,out = torch.max(p, dim=1)

        total += imgs.shape[0]
        count += (out == labels).sum()

    print(f"accurancy : {count / total}")

In [ ]:
resnet = torchvision.models.resnet18()
print(resnet)